Install Dependencies

In [ ]:
!pip install -q bitsandbytes accelerate peft transformers datasets trl fsspec==2025.3.2 -U

Convert Dataset for Training

In [ ]:
import json

input_file = "training_data_multiturn.jsonl"
output_file = "training_data_for_lora.jsonl"

with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
    for line in f_in:
        data = json.loads(line)
        converted = {
            "instruction": data["prompt"].strip(),
            "output": data["response"].strip()
        }
        json.dump(converted, f_out, ensure_ascii=False)
        f_out.write("\\n")

print(f"✅ Converted to {output_file}")


Load Converted Dataset

In [ ]:
from datasets import load_dataset

# Load dataset in instruction/output format
dataset = load_dataset("json", data_files="training_data_for_lora.jsonl")

# Optional: create train/test split (5% for eval)
dataset = dataset["train"].train_test_split(test_size=0.05)

# Preview the structure
dataset


Log in to Hugging Face

In [ ]:
from huggingface_hub import login

login()


Load Model + Tokenizer (4-bit Quantized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Load base model + tokenizer
model_name = "mistralai/Mistral-7B-v0.1"

# Use bfloat16 if supported, else fallback to float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = dict(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", **bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Prepare for LoRA fine-tuning
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✅ Model loaded, quantized, and LoRA-ready.")


Train Model with LoRA

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

steps_per_epoch = len(dataset["train"]) // (1 * 4)

training_args = TrainingArguments(
    output_dir="gurt-mistral-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    eval_steps=steps_per_epoch,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="instruction",  # <-- updated field
    args=training_args
)

trainer.train()


Save LoRA Adapter & Tokenizer + Export to Drive

In [ ]:
# Define where to save in Colab
save_dir = "/content/gurt-mistral-lora"

# Save LoRA adapter and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"✅ Model and tokenizer saved to: {save_dir}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy to Google Drive
!cp -r {save_dir} /content/drive/MyDrive/
print("✅ Backup saved to Google Drive at: /MyDrive/gurt-mistral-lora")


(If Necessary) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


(If Necessary) Load Model from Drive

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Load from Drive
model_path = "/content/drive/MyDrive/mistral-whatsapp-final"

# 4-bit quantization setup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, model_path)


Generating a Response

In [ ]:
import torch

def chat_response(prompt, max_new_tokens=120):
    # Append bot name if needed
    prompt += "\nGurt:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract Gurt's reply
    if "Gurt:" in decoded:
        reply = decoded.split("Gurt:")[-1].strip().split("\n")[0]
    else:
        reply = decoded[len(prompt):].strip().split("\n")[0]

    return reply


Testing

In [ ]:
prompt = """Petr: alain or casper?"""

response = chat_response(prompt)
print("Generated Response:", response)
